In [10]:
# (Optional) If needed in a fresh Colab runtime, uncomment:
# !pip install -q scikit-image opencv-python-headless

import os
from pathlib import Path
import numpy as np
import torch

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# Reproducibility
def set_seed(seed=42):
    import random
    import numpy as np
    import torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Project paths
PROJECT_ROOT = Path("/Users/sangsun/face-id-project")
PROC_DIR     = PROJECT_ROOT / "data" / "processed"   # expects train/val/test subfolders
FEAT_DIR     = PROJECT_ROOT / "features"             # where .npy features will be saved
FEAT_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [11]:
# Shared image size
IMG_SIZE = (160, 160)

# Training-time augmentation (for end-to-end CNN training pipelines)
train_tfms = transforms.Compose([
    transforms.RandomHorizontalFlip(),                         # pose invariance
    transforms.RandomRotation(5),                              # small-angle robustness
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),      # lighting robustness
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)                     # roughly [-1, 1]
])

# Evaluation transforms (deterministic) — used for feature extraction
eval_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])


In [13]:
# Remove class folders that have no valid image files
from pathlib import Path

ROOT = Path("/Users/sangsun/face-id-project/data/processed")  # ABSOLUTE path
valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp", ".ppm", ".pgm"}

def prune_empty_classes(split_dir: Path):
    removed = []
    for cls_dir in split_dir.iterdir():
        if not cls_dir.is_dir():
            continue
        has_img = any(p.suffix.lower() in valid_ext for p in cls_dir.rglob("*"))
        if not has_img:
            # remove the empty class dir
            try:
                cls_dir.rmdir()  # only works if truly empty
            except OSError:
                # if it contains non-image junk, remove all files then the dir
                for p in cls_dir.rglob("*"):
                    if p.is_file(): p.unlink()
                for p in sorted(cls_dir.rglob("*"), reverse=True):
                    if p.is_dir(): p.rmdir()
                cls_dir.rmdir()
            removed.append(cls_dir.name)
    return removed

removed_train = prune_empty_classes(ROOT / "train")
removed_val   = prune_empty_classes(ROOT / "val")
removed_test  = prune_empty_classes(ROOT / "test")

print(f"Removed {len(removed_train)} empty classes from train")
print(f"Removed {len(removed_val)} empty classes from val")
print(f"Removed {len(removed_test)} empty classes from test")


Removed 0 empty classes from train
Removed 1424 empty classes from val
Removed 1424 empty classes from test


In [14]:
# Build ImageFolder datasets (person-wise folder structure)
train_ds = datasets.ImageFolder(PROC_DIR / "train", transform=eval_tfms)
val_ds   = datasets.ImageFolder(PROC_DIR / "val",   transform=eval_tfms)
test_dir = PROC_DIR / "test"
test_ds  = datasets.ImageFolder(test_dir, transform=eval_tfms) if test_dir.exists() else None

num_classes = len(train_ds.classes)
print(f"Classes: {num_classes} | train={len(train_ds)} val={len(val_ds)}",
      f"test={len(test_ds) if test_ds is not None else 0}")

# Efficient, deterministic loaders for feature extraction
loader_kwargs = dict(batch_size=128, shuffle=False, num_workers=2,
                     pin_memory=(device.type == "cuda"))
train_loader = DataLoader(train_ds, **loader_kwargs)
val_loader   = DataLoader(val_ds,   **loader_kwargs)
test_loader  = DataLoader(test_ds,  **loader_kwargs) if test_ds is not None else None


Classes: 1680 | train=7836 val=664 test=664


In [16]:
# Fixed: MobileNetV2 embeddings without using a non-existent `avgpool` attribute
import torch, numpy as np
from torch.utils.data import DataLoader
from torchvision import models, datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# (Recommended) Use ImageNet normalization for pretrained weights
IMG_SIZE = (160, 160)
eval_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(PROC_DIR/'train', transform=eval_tfms)
val_ds   = datasets.ImageFolder(PROC_DIR/'val',   transform=eval_tfms)
test_ds  = datasets.ImageFolder(PROC_DIR/'test',  transform=eval_tfms)

loader_k = dict(batch_size=128, shuffle=False, num_workers=2, pin_memory=(device.type=='cuda'))
train_loader = DataLoader(train_ds, **loader_k)
val_loader   = DataLoader(val_ds,   **loader_k)
test_loader  = DataLoader(test_ds,  **loader_k)

# Pretrained MobileNetV2 as a frozen feature extractor
weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
net = models.mobilenet_v2(weights=weights).to(device).eval()

@torch.no_grad()
def extract_embeddings(dl):
    """
    Pass images through `net.features`, then apply global average pooling
    to get a [B, 1280] embedding (MobileNetV2 width = 1280).
    """
    feats, labels = [], []
    for x, y in dl:
        x = x.to(device)
        # Features: [B, 1280, H', W']
        f = net.features(x)
        # Global average pooling -> [B, 1280]
        f = f.mean(dim=(2, 3))
        feats.append(f.cpu().numpy())
        labels.append(y.numpy())
    return np.vstack(feats).astype('float32'), np.concatenate(labels)

X_train, y_train = extract_embeddings(train_loader)
X_val,   y_val   = extract_embeddings(val_loader)
X_test,  y_test  = extract_embeddings(test_loader)

print("Embeddings:", X_train.shape, X_val.shape, X_test.shape)  # e.g., (N, 1280)


Embeddings: (7836, 1280) (664, 1280) (664, 1280)
